# Sky Trails — ADS-B Data Router Prototype

This notebook collects live ADS-B aircraft messages from the websocket stream.

The goal is to turn individual aircraft packets into clean trail points that can later be used for our map visualization.

## 1. Imports

This section loads the Python libraries we need.

- `connect` lets us connect to the live websocket stream.
- `json` lets us read and write JSON data.

In [ ]:
from websockets.sync.client import connect
import json

In this example we use the websockets library (https://websockets.readthedocs.io/en/stable/)

This is a super simple way to receive the websocket data, but feel free to use other methods

## 2. Settings

These settings control the main parts of the notebook.

Keeping them here makes it easier to change the websocket address, the number of messages we collect, and the output file name without searching through the code.

In [ ]:
ADSB_WEBSOCKET_URL = "ws://192.87.172.82:1338"

MAX_MESSAGES = 50

OUTPUT_FILE = "trail_points.json"

## 3. Packet Format

The websocket gives us one JSON entry for each ADS-B packet received by one of the campus receivers.

A single packet does **not** always contain all aircraft information.
This is why we combine packets with the same aircraft `address` later in the notebook.

### 3.1 Possible fields

| Field | Meaning |
|---|---|
| `address` | The aircraft address. We use this as the plane ID. |
| `altitude` | The aircraft altitude, in feet. |
| `latitude` | The north/south map position of the aircraft. |
| `longitude` | The east/west map position of the aircraft. |
| `speed` | The aircraft speed, likely in knots. |
| `heading` | The direction the aircraft is moving, in degrees. |
| `callsign` | The flight/aircraft callsign, when available. |
| `timestamp` | The time when the receiver received the packet. |
| `rssi` | Signal strength received by the antenna. |
| `receiver` | The receiver that heard the packet, either `zi-5067` or `zi-5110`. |

### 3.2 Two receivers

Because there are two receivers, the same aircraft packet may sometimes appear twice.

When this happens, most aircraft fields may be identical, but fields such as `timestamp`, `rssi`, and `receiver` can differ. This is useful for comparing how each receiver hears the same aircraft.

### 3.3 Different packet types

There are different kinds of ADS-B messages. Each kind gives us a different puzzle piece.

| Message type | Usually contains |
|---|---|
| Location message | `latitude` and `longitude` |
| Altitude message | `altitude` |
| Speed message | `speed` and `heading` |
| Callsign message | `callsign` |

Because these pieces can arrive separately, the notebook uses `plane_memory` to remember the latest known information for each aircraft.

## 4. Data Storage

Before we receive live data, we create two empty containers where useful aircraft information can be stored.

We use two containers:

- `plane_memory` remembers the latest known information for each aircraft.
- `trail_points` stores drawable map points over time.

In [ ]:
plane_memory = {}
trail_points = []

## 5. Message Handling

This function decides what to do with one incoming ADS-B message.

It uses the aircraft `address` to update `plane_memory`.
If the message contains a drawable position, it also saves a point into `trail_points`.

In simple words: this is where one small packet becomes part of a plane trail.

In [ ]:
def handle_message(msg, seen_addresses):
    address = msg.get("address")

    # If the message has no aircraft address, we cannot group it.
    if address is None:
        return

    # If this is the first time we see this aircraft,
    # create a little memory entry for it.
    if address not in plane_memory:
        plane_memory[address] = {
            "address": address,
            "speed": None,
            "heading": None,
            "altitude": None,
            "latitude": None,
            "longitude": None,
            "timestamp": None,
            "rssi": None,
            "receiver": None
        }

    # Update only the fields that are present in this message.
    for field in [
        "speed",
        "heading",
        "altitude",
        "latitude",
        "longitude",
        "timestamp",
        "rssi",
        "receiver"
    ]:
        if field in msg:
            plane_memory[address][field] = msg[field]

    # Only print planes that are useful for drawing a map.
    has_position = (
        plane_memory[address]["latitude"] is not None
        and plane_memory[address]["longitude"] is not None
    )

    if has_position:
        point = {
            "address": address,
            "latitude": plane_memory[address]["latitude"],
            "longitude": plane_memory[address]["longitude"],
            "altitude": plane_memory[address]["altitude"],
            "speed": plane_memory[address]["speed"],
            "heading": plane_memory[address]["heading"],
            "timestamp": plane_memory[address]["timestamp"],
            "rssi": plane_memory[address]["rssi"],
            "receiver": plane_memory[address]["receiver"]
        }

        trail_points.append(point)

    if has_position and address not in seen_addresses:
        print("Plane memory now has a drawable plane:")
        print(json.dumps(plane_memory[address], indent=2))
        print()
        seen_addresses.add(address)

## 6. Receive Live ADS-B Data

This function opens the websocket connection and listens for incoming ADS-B packets.

For each message, it:
1. receives a raw websocket message,
2. turns it from JSON text into a Python dictionary,
3. sends it to `handle_message()` for processing,
4. repeats this until `MAX_MESSAGES` messages have been received.

This function is the “river gate” of the notebook: it lets live aircraft data flow into our processing code.

In [ ]:
def receive_data():
    max_msg =  MAX_MESSAGES

    # This resets every time you run receive_data()
    seen_addresses = set()

    with connect(ADSB_WEBSOCKET_URL) as websocket:
        count = 0

        while count < max_msg:
            msg = websocket.recv()

            try:
                msg = json.loads(msg)
                handle_message(msg, seen_addresses)

            except json.JSONDecodeError:
                print("Failed to decode json, assuming next packet will be ok...")

            except Exception as e:
                print("Something went horribly wrong!")
                print("Error:", e)
                break

            count += 1

        print("Received %d messages!" % count)

## 7. Run Data Collection

This cell starts the live data collection process.

It calls `receive_data()`, which connects to the ADS-B websocket, receives messages, and sends each message to `handle_message()`.

After this cell runs, `plane_memory` and `trail_points` should contain collected aircraft data.

In [ ]:
receive_data()

## 8. Inspect Collected Trail Points

This section checks what we collected after running the live data collection.

`len(trail_points)` tells us how many drawable map points we saved.
`trail_points[:3]` shows the first three points, so we can quickly check whether the data looks correct.

In [ ]:
len(trail_points)
trail_points[:3]

## 9. Save Trail Points for Visualization

The collected trail points currently live only in notebook memory.

This step saves them into a JSON file, so the visualization part of the project can load the data without needing to connect to the live websocket every time.

In [ ]:
# Save collected trail points as a JSON file.
# This turns our temporary notebook data into a reusable file.

with open(OUTPUT_FILE, "w") as file:
    json.dump(trail_points, file, indent=2)

print(f"Saved {len(trail_points)} trail points to {OUTPUT_FILE}")

## 10. Check Saved Trail Points

This section reads the saved JSON file back into Python.

If this works, we know that `trail_points.json` was created correctly and can be used by the visualization part of the project.

This is a small safety check before moving on to the map.

In [ ]:
# Read the saved trail points back from the JSON file.
# This checks whether the file was created correctly.

with open(OUTPUT_FILE, "r") as file:
    saved_trail_points = json.load(file)

print(f"Loaded {len(saved_trail_points)} trail points from {OUTPUT_FILE}")

saved_trail_points[:3]

## 11. First Map Prototype

This section creates a simple interactive map from the saved trail points.

For now, each aircraft position is shown as a dot. Later, we can connect dots with the same `address` to create trails.

In [ ]:
# Create a first simple map from the saved trail points.

import folium

# Keep only points that have a valid map position.
valid_points = [
    point for point in saved_trail_points
    if point["latitude"] is not None and point["longitude"] is not None
]

# Use the average position as the center of the map.
center_latitude = sum(point["latitude"] for point in valid_points) / len(valid_points)
center_longitude = sum(point["longitude"] for point in valid_points) / len(valid_points)

plane_map = folium.Map(
    location=[center_latitude, center_longitude],
    zoom_start=7
)

# Add a dot for each saved aircraft position.
for point in valid_points:
    popup_text = f"""
    Aircraft: {point["address"]}<br>
    Altitude: {point["altitude"]}<br>
    Speed: {point["speed"]}<br>
    Heading: {point["heading"]}<br>
    Receiver: {point["receiver"]}<br>
    RSSI: {point["rssi"]}
    """

    folium.CircleMarker(
        location=[point["latitude"], point["longitude"]],
        radius=4,
        popup=popup_text,
        fill=True
    ).add_to(plane_map)

plane_map.save("first_plane_map.html")

print("Saved map to first_plane_map.html")

plane_map

## 12. Group Trail Points by Aircraft

This section groups saved trail points by aircraft `address`.

A trail can only be drawn when the same aircraft has at least two position points.

In [12]:
# Group trail points by aircraft address.
# Each aircraft will get its own list of map points.

points_by_aircraft = {}

for point in saved_trail_points:
    address = point["address"]

    if address not in points_by_aircraft:
        points_by_aircraft[address] = []

    points_by_aircraft[address].append(point)

print(f"Found {len(points_by_aircraft)} different aircraft.")

for address, points in points_by_aircraft.items():
    print(address, "has", len(points), "points")

Found 6 different aircraft.
4080E9 has 19 points
3C4B49 has 5 points
3D48FE has 7 points
010246 has 7 points
4D207C has 5 points
48663E has 1 points
